# N.L Engine 워크플로우 실행 흐름 가이드

`main.py`의 UI 영역을 제외하고 **쿼리 매핑 → 서사 에이전트 실행 → 그래프 매니저**로 데이터가 어떻게 넘어가는지 단계별로 볼 수 있는 테스트 노트북입니다.

In [ ]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('.'))
load_dotenv()
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

print("✅ 환경 변수 및 라이브러리 경로 설정 완료")

## 1. 사용자의 초기 아이디어 입력

In [ ]:
user_idea = "어느 날 뒷산에서 발견된 낡은 보물지도로 인해 일상이 무너진 평범한 소년의 이야기."
theory_type = "THEORY_PROPP_VOGLER_HYBRID"

print("사용자 쿼리:", user_idea)

## 2. Mapper Agent를 통한 시작 노드(Milestone) 매핑
`main.py`의 `mapper.map_input_to_node()` 동작 테스트

In [ ]:
from src.mapper_agent import MapperAgent

mapper = MapperAgent(theory_type=theory_type)
start_node_id = mapper.map_input_to_node(user_idea)

print(f"🎯 판별된 시작 노드(Milestone): {start_node_id}")

## 3. 서사 에이전트(LangGraph) 상태 초기화 및 빌드
에이전트에게 정보를 넘기기 전 `initial_state`를 정의하는 부분입니다.

In [ ]:
from src.narrative_agent import build_narrative_graph
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [HumanMessage(content=user_idea)],
    "current_section": "Section 01의 step_num:1",
    "is_section_complete": False,
    "idea_note": [],
    "master_data": {
        "characters": [],
        "worldview": {"genre": "Fantasy", "rules": "마법이 사라진 세계"},
        "plot_nodes": [start_node_id],
        "theory_type": theory_type
    },
    "validation_status": {},
    "next_node": "history",
    "missing_info": []
}

workflow = build_narrative_graph()

## 4. LangGraph 부분 실행 (스트리밍 파싱)
`main.py`에서는 `.invoke()`로 한 번에 처리하지만, 어떤 노드를 거쳐가는지 흐름을 보기 위해 `.stream()` 메서드를 사용해 각 노드의 진행(History -> Router -> Generator -> 검증) 과정을 추적합니다.

In [ ]:
config = {"configurable": {"thread_id": "test_user_01"}}
print("🚀 서사 에이전트 실행 시작...\n")

for chunk in workflow.stream(initial_state, config):
    for node_name, output in chunk.items():
        print(f"🔵 [진입한 노드: {node_name}]")
        if "messages" in output and len(output["messages"]) > 0:
            print(f"> AI 응답:\n{output['messages'][-1].content}\n")
        if "next_node" in output:
            print(f"> 라우팅 상태 -> {output['next_node']}\n")
        print("-" * 40)

## 5. 생성된 서사를 바탕으로 그래프 시각화 (NetworkX 연동)
최종적으로 반환된 상태를 `NarrativeGraphManager`에 등록하고 데이터를 반환받아 시각화를 준비합니다.

In [ ]:
from src.graph_manager import NarrativeGraphManager
import json

manager = NarrativeGraphManager()

# 예시용 데이터로 그래프 삽입
manager.add_milestone("Start")
manager.add_milestone(start_node_id)
manager.add_causality("Start", start_node_id, "사건의 발단")

graph_data = manager.get_graph_data()
print("📊 시각화용 그래프 구조 추출:")
print(json.dumps(graph_data, ensure_ascii=False, indent=2))